# ACCESS-IoT · Notebook 3 — telemetry effect, transfer between ACCESS-IoT builds (E6), explanation ground truth (E8)
- **E6a** telemetry ablation: summarised from Notebook 2 (main = CICIoMT2024, no_telemetry, wustl_telemetry) plus device-compromise detection.
- **E6b** transfer: a history-aware gradient-boosted model trained on ACCESS-IoT-Sim (the `no_telemetry` build: fully simulated,
  no real traffic) and tested on ACCESS-IoT (the `main` build: CICIoMT2024 telemetry), and the reverse
  (real-traffic inputs set to neutral training medians when absent).
- **E8** explanation ground truth: do SHAP attributions of the history-aware model point to the attribute groups that actually
  generated each malicious request (released as ground truth)?
- Fig. 2 panel d.
Input: the released builds in `DATA_DIR/builds/<build>/`. Cell X1 also needs the result tables of Notebook 2
(`E4_T1_mean.csv`, `E7_per_threat_MGR.csv` in REPORTS): run Notebook 2 first, or copy them from the `results` zip on Zenodo.
Runtime: a GPU is recommended (XGBoost); without one everything runs on the CPU, more slowly.

In [ ]:
# X0 — settings and helpers
import os, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
# Data: unzip every ACCESS-IoT-v1.0_<build>.zip from Zenodo (https://doi.org/10.5281/zenodo.23838631) into DATA_DIR,
# which gives DATA_DIR/builds/<build>/requests.parquet, ... . Outputs (tables, figures) go to REPORTS.
# On Colab: from google.colab import drive; drive.mount("/content/drive") and set DATA_DIR to the folder on your Drive.
DATA_DIR = Path(os.environ.get("ACCESS_IOT_DATA", "data"))
REPORTS = Path(os.environ.get("ACCESS_IOT_REPORTS", "reports"))
REL, REP = DATA_DIR / "builds", REPORTS
REP.mkdir(parents=True, exist_ok=True)
assert REL.exists(), f"no builds found in {REL}: download and unzip the Zenodo files first"
import xgboost as xgb, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
P, L, D, B = 0, 1, 2, 3
ENC = {"PERMIT": 0, "LIMITED": 1, "DENY": 2, "BREAK_GLASS": 3}
VISIBLE_KINDS = ["request", "pip", "telemetry", "baseline"]
ID_COLS = {"RequestID", "Timestamp", "SessionID", "UserID", "PatientID", "DeviceID", "DelegationID", "DelegatorID", "DelegateeID"}

def load(b):
    d = pd.read_parquet(REL / b / "requests.parquet")
    for c in ("Timestamp", "FeedbackTime"): d[c] = pd.to_datetime(d[c])
    return d

def features(b):
    d, dic = load(b), pd.read_csv(REL / b / "data_dictionary.csv")
    cols = [c for c in dic.loc[dic.Visibility.isin(VISIBLE_KINDS), "Column"] if c in d and c not in ID_COLS]
    X = d[cols].copy()
    X["HourOfDay"] = d.Timestamp.dt.hour; X["HasDelegation"] = d.DelegationID.notna().astype(int)
    X["DeviceDeptMatchesUser"] = (d.DeviceDepartment == d.UserDepartment).astype(int)
    H = pd.read_parquet(REL / b / "features/history.parquet"); H.index = X.index
    X = pd.concat([X, H], axis=1)
    mask = np.load(REL / b / "views/rl_admissible_mask.npy")
    return d, X, mask

def model(seed=0, multi=True):
    p = dict(n_estimators=500, learning_rate=0.06, max_depth=8, subsample=0.9, colsample_bytree=0.8, tree_method="hist",
             device=DEVICE, enable_categorical=True, max_cat_to_onehot=1, random_state=seed)
    return xgb.XGBClassifier(objective="multi:softprob", **p) if multi else xgb.XGBClassifier(**p)

def align(Xs, Xt, cats_from):
    """same columns and category levels as the training frame; missing real-traffic inputs set to neutral training medians"""
    Xt = Xt.copy()
    for c in Xs.columns:
        if c not in Xt: Xt[c] = Xs[c].median() if pd.api.types.is_numeric_dtype(Xs[c]) else Xs[c].mode().iloc[0]
    Xt = Xt[Xs.columns]
    for c in Xs.columns:
        if not pd.api.types.is_numeric_dtype(Xs[c]) or str(Xs[c].dtype) == "category":
            Xt[c] = pd.Categorical(Xt[c].astype(str), categories=cats_from[c].cat.categories)
    return Xt

def cat(X):
    X = X.copy()
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]): X[c] = X[c].astype(object).fillna("NA").astype(str).astype("category")
    return X

def t1(a, d, mask, rows):
    o = d.OptimalDecision.map(ENC).to_numpy()[rows]; m = d.IsMalicious.to_numpy()[rows] == 1; a = a[rows]; g = np.isin(a, [P, L, B])
    return {"MacroF1": round(f1_score(o, a, labels=[0, 1, 2, 3], average="macro", zero_division=0), 4), "MGR": round(g[m].mean(), 4),
            "LDR": round((a[o == P] == D).mean(), 4), "EDR": round((a[o == B] == D).mean(), 4)}

In [ ]:
# X1 — E6a: telemetry ablation (summary of Notebook 2) and device-compromise detection by the network summary
T1 = pd.read_csv(REP / "E4_T1_mean.csv", index_col=[0, 1]); T23 = pd.read_csv(REP / "E4_T2_T3_mean.csv", index_col=[0, 1])
tele = [b for b in ["main", "no_telemetry", "wustl_telemetry"] if b in T1.index.get_level_values(0)]
keep = ["Supervised GBDT, with history", "Supervised GBDT, single request", "Risk-adaptive", "Zero-trust re-evaluation",
        "Insider detection (Isolation Forest)", "Delayed-reward neural bandit"]
rows = []
for b in tele:
    d = load(b, ) if False else pd.read_parquet(REL / b / "requests.parquet", columns=["Split", "CompromisedPath", "ThreatType"] +
                                                 (["NetAnomaly"] if b != "no_telemetry" else []))
    te = d.Split == "test"; dev = ((d.CompromisedPath == 1) | (d.ThreatType == "virtual_object_tampering"))[te]
    auc = roc_auc_score(dev, d.NetAnomaly[te]) if "NetAnomaly" in d else np.nan
    for m in keep:
        if (b, m) in T1.index:
            rows.append({"telemetry": {"main": "ACCESS-IoT: CICIoMT2024 (real)", "no_telemetry": "ACCESS-IoT-Sim: none (simulated only)",
                                       "wustl_telemetry": "ACCESS-IoT-WUSTL: WUSTL-EHMS-2020 (real)"}[b], "method": m,
                         **T1.loc[(b, m), ["MacroF1", "MGR", "LDR", "EDR"]].round(3).to_dict(),
                         "T7 tampering MGR": round(pd.read_csv(REP / "E7_per_threat_MGR.csv", index_col=[0, 1]).loc[(b, m)].get("virtual_object_tampering", np.nan), 3),
                         "NetAnomaly AUC (device compromise)": round(auc, 3)})
e6a = pd.DataFrame(rows); e6a.to_csv(REP / "E6a_telemetry.csv", index=False); print(e6a.to_string(index=False))

In [ ]:
# X2 — E6b: transfer between ACCESS-IoT-Sim (no_telemetry build, no real traffic) and ACCESS-IoT (main build, CICIoMT2024)
dS, XS, mS = features("no_telemetry"); dR, XR, mR = features("main")
XS, XR = cat(XS), cat(XR)
res = []
for (src, dsrc, Xsrc), (tgt, dtgt, Xtgt, mtgt) in [
        (("ACCESS-IoT-Sim (no telemetry)", dS, XS), ("ACCESS-IoT-Sim (no telemetry)", dS, XS, mS)), (("ACCESS-IoT-Sim (no telemetry)", dS, XS), ("ACCESS-IoT (CICIoMT2024 telemetry)", dR, XR, mR)),
        (("ACCESS-IoT (CICIoMT2024 telemetry)", dR, XR), ("ACCESS-IoT (CICIoMT2024 telemetry)", dR, XR, mR)), (("ACCESS-IoT (CICIoMT2024 telemetry)", dR, XR), ("ACCESS-IoT-Sim (no telemetry)", dS, XS, mS))]:
    tr = (dsrc.Split == "train").to_numpy(); te = (dtgt.Split == "test").to_numpy()
    opt = dsrc.OptimalDecision.map(ENC).to_numpy()
    for sd in (0, 1, 2):
        m = model(sd).fit(Xsrc[tr], opt[tr])
        Xe = align(Xsrc, Xtgt, Xsrc)
        pr = m.predict_proba(Xe); a = np.where(mtgt, pr, -1).argmax(1)
        mb = model(sd, False).fit(Xsrc[tr], dsrc.IsMalicious.to_numpy()[tr]); s = mb.predict_proba(Xe)[:, 1]
        y = dtgt.IsMalicious.to_numpy()[te]
        res.append({"train on": src, "test on": tgt, "seed": sd, **t1(a, dtgt, mtgt, te),
                    "malicious PR-AUC": round(average_precision_score(y, s[te]), 4)})
e6b = pd.DataFrame(res).groupby(["train on", "test on"], sort=False).agg(["mean", "std"]).drop(columns="seed").round(3)
e6b.to_csv(REP / "E6b_transfer.csv"); print(e6b.to_string())

In [ ]:
# X3 — E8: do attributions recover the generating cause? (SHAP of the history-aware malicious-request model, main build, GPU)
d, X, mask = features("main"); X = cat(X)
tr, te = (d.Split == "train").to_numpy(), (d.Split == "test").to_numpy(); y = d.IsMalicious.to_numpy()
mb = model(0, False).fit(X[tr], y[tr])
rows = np.flatnonzero(te & (y == 1))
dm = xgb.DMatrix(X.iloc[rows], enable_categorical=True)
contrib = mb.get_booster().predict(dm, pred_contribs=True)[:, :-1]          # last column = bias
cols = list(X.columns)
def group(c):
    if c.startswith("NF_") or c.startswith("dev_") or c in {"NetAnomaly", "PacketLoss", "NetworkLoad", "ResponseDelay", "Availability",
        "BatteryLevel", "DeviceAuthFailures24h", "CommunicationSuccessRate", "FirmwareAttestation", "Layer", "DeviceType", "DeviceDepartment", "Location"}:
        return "device & network"
    if c in {"OnCareTeam", "SameDepartment", "PatientVIP", "u_offteam_7d", "u_new_patient", "u_newpat_24h", "u_vip_7d"}: return "patient relationship"
    if c in {"WithinShift", "HourOfDay", "u_offshift_7d"}: return "time & shift"
    if c in {"LoginFailedAttempts", "AccessChannel", "SourceNetwork", "u_new_device", "u_new_network", "DeviceDeptMatchesUser",
             "u_cnt_1h", "u_cnt_24h", "s_idx"}: return "authentication & access context"
    if c in {"HasDelegation", "DelegationDepth", "DelegationScope", "RootDelegatorMaxDepth", "deleg_pair_prior", "delegator_grants_7d",
             "delegator_prior_malicious_fb", "u_grants_7d", "RequestType"}: return "delegation"
    if c in {"ResourceType", "RequestedAction", "ResourceSensitivity", "u_write_7d"}: return "operation & resource"
    if c in {"EmergencyFlag", "HeartRate", "SpO2", "SystolicBP", "RespiratoryRate", "Temperature", "BloodGlucose", "AcuityScore", "u_emerg_30d"}:
        return "clinical state & emergency"
    if c in {"ABACDecision", "StaticBreakGlassDecision", "CapabilityCheck"}: return "policy outputs"
    if c in {"u_prior_malicious_fb", "u_denied_7d"}: return "revealed track record"
    return "user & patient profile"
G = sorted({group(c) for c in cols}); gi = np.array([G.index(group(c)) for c in cols])
gabs = np.zeros((len(rows), len(G)))
for j in range(len(G)): gabs[:, j] = np.abs(contrib[:, gi == j]).sum(1)
CAUSE = {"gradual_decay": {"patient relationship", "time & shift"},
         "bursty_attack": {"authentication & access context"},
         "collusion": {"delegation", "patient relationship"},
         "delegation_misuse": {"delegation", "operation & resource", "time & shift"},
         "privilege_escalation": {"policy outputs", "operation & resource", "delegation"},
         "emergency_misuse": {"clinical state & emergency", "patient relationship"},
         "virtual_object_tampering": {"device & network", "operation & resource"}}
th = d.ThreatType.to_numpy()[rows]; top = np.argsort(-gabs, 1)
from math import comb
out = []
for t, causes in CAUSE.items():
    sel = th == t
    if not sel.any(): continue
    k = len(causes & set(G)); n = len(G)
    for K in (1, 2, 3):
        hit = np.array([bool({G[j] for j in r[:K]} & causes) for r in top[sel]]).mean()
        rnd = 1 - comb(n - k, K) / comb(n, K)
        out.append({"threat": t, "top-K groups": K, "hit rate": round(hit, 3), "random expectation": round(rnd, 3), "n": int(sel.sum())})
e8 = pd.DataFrame(out); e8.to_csv(REP / "E8_explanation_ground_truth.csv", index=False)
print(e8.pivot_table(index="threat", columns="top-K groups", values=["hit rate", "random expectation"]).to_string())

In [ ]:
# X4 — Fig. 2 panel d: what real telemetry adds - anomaly-score AUC for device compromise and overall malicious-grant rate
#      (history-aware supervised model) per telemetry source. T7 tampering MGR is not plotted: only 42 test requests.
import matplotlib.pyplot as plt
sub = e6a[e6a.method == "Supervised GBDT, with history"].reset_index(drop=True)
short = lambda t: next(v for k, v in {"Sim": "ACCESS-IoT-Sim\n(no real traffic)", "WUSTL": "ACCESS-IoT-WUSTL\n(WUSTL-EHMS-2020)",
                                      "CICIoMT": "ACCESS-IoT\n(CICIoMT2024)"}.items() if k in t)
fig, ax = plt.subplots(figsize=(4.6, 3.6)); x = np.arange(len(sub)); w = 0.38
auc = sub["NetAnomaly AUC (device compromise)"].to_numpy(float)
ax.bar(x - w / 2, np.nan_to_num(auc), w, label="anomaly score AUC (device compromise)", color="#9aa5b1")
ax.bar(x + w / 2, sub["MGR"], w, label="malicious-grant rate (all threats)", color="#1f4e79")
for i_, v in enumerate(auc):
    if np.isnan(v): ax.text(i_ - w / 2, 0.02, "n/a", ha="center", fontsize=7)
ax.axhline(0.5, ls=":", c="grey", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels([short(t) for t in sub.telemetry], fontsize=7); ax.set_ylim(0, 1)
ax.legend(fontsize=7, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=1)
ax.spines[["top", "right"]].set_visible(False); ax.set_title("(d) device telemetry", fontsize=9); ax.tick_params(labelsize=7)
fig.tight_layout(); fig.savefig(REP / "fig2_validity_d.pdf", bbox_inches="tight"); fig.savefig(REP / "fig2_validity_d.png", dpi=200, bbox_inches="tight"); plt.show()
print("All experiment outputs are in", REP)